# Notebook 03: Text preprocessing

This notebook applies the 12 cleaning rules (the 11 from notebook 02 plus the local-language glossary), builds the experiment splits, and splits reviews into
sentences and clauses for aspect-level analysis. The code lives in `src/absa/text.py` and `src/absa/data.py`, so the
Streamlit app uses exactly the same preprocessing as the experiments.

Course topics covered: regular expressions, tokenisation, normalisation (case, contractions, special characters,
stopwords), lemmatisation, POS tagging and dependency parsing.

In [1]:
# Setup
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from absa.setup import ensure_resources
from absa import viz
ensure_resources()
viz.use_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

In [2]:
from absa.text import clean, tokenize, remove_stopwords, split_sentences, split_clauses, lemmatize
from absa.data import load_raw, build_reviews, build_sentences

raw = load_raw()
print(f"{len(raw)} reviews after removing empty ones (rule 1)")

6206 reviews after removing empty ones (rule 1)


## 1. The cleaning function, rule by rule

`clean()` applies the rules in a fixed order. Order matters: contractions must be expanded *before* punctuation is
removed, otherwise *doesn't* becomes *doesn t* and the negation is lost.

In [3]:
examples = [
    "It doesn't work!!! EcoCash is useless 😡",
    "Pls fix the app, it nolonger opens. Goood app tho",
    "Innbucks says there is no transaction fees but it charges me. Otherwise I'm happy",
    "my zb app is better than CBZ Touch and lnnbucks",
    "I cant login since the update... App sends OTP but won't open keyboard",
    "call me on 0772 123 456 if you can help",
    "hmmm panhau dze limit apa asi the app is quite fast",
    "App iyi chakadhakwa, haishande kana ndaupdater",
]
pd.DataFrame({"original": examples, "cleaned": [clean(t) for t in examples]})

,original,cleaned
0,It doesn't work!!! EcoCash is useless 😡,it does not work ! brandname is useless emj_enraged_face
1,"Pls fix the app, it nolonger opens. Goood app tho",please fix the app it no longer opens good app tho
2,Innbucks says there is no transaction fees but it charges me. Otherwise I'm happy,brandname says there is no transaction fees but it charges me otherwise i am happy
3,my zb app is better than CBZ Touch and lnnbucks,brandname app is better than brandname and brandname
4,I cant login since the update... App sends OTP but won't open keyboard,i cannot login since the update app sends otp but will not open keyboard
5,call me on 0772 123 456 if you can help,call me on phonetoken if you can help
6,hmmm panhau dze limit apa asi the app is quite fast,hmm panhau dze limit apa asi the app is quite fast
7,"App iyi chakadhakwa, haishande kana ndaupdater",app iyi chakadhakwa malfunctioning haishande not working kana ndaupdater


What each change shows:
* `doesn't -> does not`, `cant -> cannot`, `won't -> will not`: the negation survives (rule 3)
* `Pls -> please`, `nolonger -> no longer`, `Goood -> good` (rules 4-6)
* every brand -> `brandname` (rule 7), including misspellings such as *lnnbucks*
* `0772 123 456 -> phonetoken` (rule 10)
* `😡 -> emj_enraged_face`: emojis become words, so the classifier can learn from them
* `!!!` -> a single `!` token; `!` and `?` are kept because they carry emotion

Rule 12 (added after testing the interface): known Shona/Ndebele sentiment words get an English gloss
*appended*, e.g. `chakadhakwa -> chakadhakwa malfunctioning`, `haishande -> haishande not working`. Without it, the
classifier (trained mostly on English) read *"App iyi chakadhakwa, haishande"* as positive: it had almost never seen
these words. The glossary (`data/lexicons/normalisation.json`) lists 25 words found in the corpus.

### How much does normalisation shrink the vocabulary?

In [4]:
from collections import Counter
raw_vocab = Counter(w for t in raw["content"] for w in t.lower().split())
cleaned = raw["content"].map(clean)
clean_vocab = Counter(w for t in cleaned for w in tokenize(t))
print(f"distinct 'words' before: {len(raw_vocab):,}   after cleaning: {len(clean_vocab):,} "
      f"({1 - len(clean_vocab) / len(raw_vocab):.0%} fewer)")

distinct 'words' before: 6,969   after cleaning: 4,557 (35% fewer)


In [5]:
variants = {
    "contractions (rule 3)": dict(contractions=False),
    "text-speak / joined / stretched (rules 4-6)": dict(slang=False),
    "brand masking (rule 7)": dict(mask_brands=False),
    "emojis to words": dict(emojis=False),
    "local-language glossary (rule 12)": dict(local=False),
}
rows = []
for name, kw in variants.items():
    changed = (raw["content"].map(lambda t: clean(t, **kw)) != cleaned).mean()
    rows.append({"rule": name, "reviews changed": f"{changed:.1%}"})
pd.DataFrame(rows)

,rule,reviews changed
0,contractions (rule 3),20.6%
1,text-speak / joined / stretched (rules 4-6),5.0%
2,brand masking (rule 7),8.7%
3,emojis to words,5.5%
4,local-language glossary (rule 12),0.3%


## 2. Tokenisation, stopwords, lemmatisation

* Tokens: words, numbers, emoji tokens, `!` and `?`.
* Stopwords: standard lists include *not, no, but*; the project list keeps them (rule 11).
* Lemmatisation (spaCy) is tested in the ablation study (notebook 05) and not used in the final model; character
  n-grams already link word forms.

In [6]:
s = clean("The app keeps crashing and the charges are not fair, I can't even log in")
toks = tokenize(s)
from absa.text import stopwords
naive_sw = set(stopwords()) | {"not", "no", "but", "cannot"}
print("tokens          :", toks)
print("standard list   :", [t for t in toks if t not in naive_sw], "  <- negation gone!")
print("our list        :", remove_stopwords(toks))
print("lemmatised      :", lemmatize([s])[0])

tokens          : ['the', 'app', 'keeps', 'crashing', 'and', 'the', 'charges', 'are', 'not', 'fair', 'i', 'cannot', 'even', 'log', 'in']
standard list   : ['app', 'keeps', 'crashing', 'charges', 'fair', 'even', 'log']   <- negation gone!
our list        : ['app', 'keeps', 'crashing', 'charges', 'not', 'fair', 'cannot', 'even', 'log']


lemmatised      : the app keep crash and the charge be not fair i can not even log in


## 3. Part-of-speech tags and dependency parsing (spaCy)

Dependency relations feed the dependency opinion scope in notebook 04, e.g. *charges* is the subject (`nsubj`) of
*are*, and *fair* its complement (`acomp`).

In [7]:
from absa.scope import nlp
from spacy import displacy
doc = nlp()(clean("The app is great but the charges are too high"))
pd.DataFrame([(t.text, t.pos_, t.dep_, t.head.text) for t in doc], columns=["token", "POS", "dependency", "head"]).T

,0,1,2,3,4,5,6,7,8,9
token,the,app,is,great,but,the,charges,are,too,high
POS,DET,NOUN,AUX,ADJ,CCONJ,DET,NOUN,AUX,ADV,ADJ
dependency,det,nsubj,ROOT,acomp,cc,det,nsubj,conj,advmod,acomp
head,app,is,is,is,is,charges,are,is,high,are


In [8]:
displacy.render(doc, style="dep", jupyter=True, options={"distance": 95, "compact": True})

## 4. Experiment splits

| subset | what | role |
|---|---|---|
| `train` | 80% of EcoCash (old), InnBucks, CBZ Touch | the model learns from these |
| `holdout` | the other 20% (stratified by label) | in-domain test |
| `test_known` | EcoCash Super App | out-of-domain test 1: new app, known brand |
| `test_unseen` | O'mari, OneMoney, MyZB, CABS | out-of-domain test 2: unseen brands |

In [9]:
reviews = build_reviews()
pd.crosstab(reviews["subset"], reviews["label"], margins=True)

label,negative,neutral,positive,All
subset,,,,
holdout,224,56,520,800
test_known,254,40,901,1195
test_unseen,490,60,464,1014
train,894,226,2077,3197
All,1862,382,3962,6206


In [10]:
pd.crosstab(reviews["subset"], reviews["label"], normalize="index").round(3)

label,negative,neutral,positive
subset,,,
holdout,0.280,0.070,0.650
test_known,0.213,0.033,0.754
test_unseen,0.483,0.059,0.458
train,0.280,0.071,0.650


Train and holdout have identical class balance (stratified split). `test_unseen` is far more negative (48%) than
training (28%): the distribution shift seen in notebook 02.

## 5. Sentences and clauses

Aspect-based analysis needs smaller units than a whole review. Reviews are typed on phones, so the splitter handles
missing spaces (*"app.Can"*), `...` used as a separator, and line breaks. Each sentence is then split at contrast
words in English, Shona (*asi*) and Ndebele (*kodwa*), where the opinion usually changes direction.

In [11]:
for t in ["Great app.Improvements still needed though.  Need emails",
          "It's a good experience. However, I can no longer access the app using my fingerprint",
          "hmmm panhau dze limit apa asi the app is quite fast"]:
    print(t)
    for snt in split_sentences(t):
        print("   sentence:", snt, "  -> clauses:", split_clauses(snt))

Great app.Improvements still needed though.  Need emails
   sentence: Great app   -> clauses: ['Great app']
   sentence: Improvements still needed though   -> clauses: ['Improvements still needed though']
   sentence: Need emails   -> clauses: ['Need emails']
It's a good experience. However, I can no longer access the app using my fingerprint
   sentence: It's a good experience   -> clauses: ["It's a good experience"]
   sentence: However, I can no longer access the app using my fingerprint   -> clauses: ['However, I can no longer access the app using my fingerprint']
hmmm panhau dze limit apa asi the app is quite fast
   sentence: hmmm panhau dze limit apa asi the app is quite fast   -> clauses: ['hmmm panhau dze limit apa', 'asi the app is quite fast']


In [12]:
sentences = build_sentences(reviews)
n_sent = sentences["sent_id"].nunique()
print(f"{len(reviews):,} reviews -> {n_sent:,} sentences -> {len(sentences):,} clauses")
print(f"sentences per review: {n_sent / reviews['reviewId'].nunique():.2f}")
print(f"sentences that contain a contrast split: {(sentences.groupby('sent_id').size() > 1).mean():.1%}")

6,206 reviews -> 8,382 sentences -> 8,904 clauses
sentences per review: 1.35
sentences that contain a contrast split: 6.0%
